# GC and PID Analysis
Brain–heart interaction analysis using Granger Causality (GC) and 
Partial Information Decomposition (PID) on EEG + ECG data.

## Imports

In [1]:
import matplotlib.pyplot as plt
import numpy as np
from braindecode.datautil.serialization import load_concat_dataset
from braindecode.datasets import BaseConcatDataset
from braindecode.preprocessing import create_fixed_length_windows
import neurokit2 as nk
from scipy.signal import detrend
from scipy.interpolate import interp1d
from scipy.signal import butter, filtfilt
from scipy.signal import hilbert
from scipy.signal import resample_poly
from scipy.stats import mannwhitneyu
from scipy.stats import ttest_ind
from scipy.stats import shapiro, normaltest
from math import gcd
import mne
from joblib import Parallel, delayed
import pickle
import pandas as pd
from pathlib import Path
import copy
import json

# Load data

In [2]:
# Load the data
OUT_PATH = r'D:\Ana_Maria\cleaned_TUH_scaled'
tuh_bhi = load_concat_dataset(OUT_PATH, preload=False)

## Electrod Definitions

Define the EEG channel list and group channels by brain region

In [38]:
eeg_ch = ('FP1', 'FP2', 'F3', 'F4', 'C3', 'C4', 'P3', 'P4', 'O1', 'O2',
    'F7', 'F8', 'T3', 'T4', 'T5', 'T6', 'FZ', 'CZ', 'PZ')

In [39]:
frontal   = ['Fp1', 'Fp2', 'F3', 'F4', 'F7', 'F8', 'Fz']
temporal  = ['T3', 'T4', 'T5', 'T6']
central   = ['C3', 'C4', 'Cz']
parietal  = ['P3', 'P4', 'Pz']
occipital = ['O1', 'O2']

## Signal Processing

Functions for envelope extraction, resampling, HRV detection and HRVband decomposition

In [ ]:

def mean_band_envelope(eeg_data, channel_indices, fs, lowcut, highcut):
    """
    Bandpass filter each requested EEG channel and compute the analytic
    amplitude envelope via the Hilbert transform, then return the mean
    envelope across all requested channels.

    Parameters
    ----------
    eeg_data        : ndarray (n_channels, n_samples) - raw EEG data
    channel_indices : list of int- which channel rows to use
    fs              : float - sampling frequency in Hz
    lowcut, highcut : float - bandpass edges in Hz

    Returns
    -------
    mean_env : ndarray (n_samples,) — average envelope across channels
    """
    envelopes = []
    for idx in channel_indices:
        signal = eeg_data[idx]
        # Bandpass filter
        nyq = fs / 2
        b, a = butter(4, [lowcut/nyq, highcut/nyq], btype='band')
        filtered = filtfilt(b, a, signal)
        # Hilbert envelope
        envelope = np.abs(hilbert(filtered))
        envelopes.append(envelope)
    
    return np.mean(envelopes, axis=0)


In [ ]:
def resample_to(signal, fs_in, fs_out):
    """
    Resample a signal from fs_in to fs_out using a polyphase filter.
    We compute the GCD of the two rates to keep the up/down factors as
    small as possible, which reduces ringing and computation time.

    Parameters
    ----------
    signal : ndarray - input signal
    fs_in  : float - original sampling rate (Hz)
    fs_out : float - target sampling rate (Hz)

    Returns
    -------
    resampled : ndarray
    """
    g = gcd(int(fs_in), int(fs_out))
    return resample_poly(signal, int(fs_out//g), int(fs_in//g))

In [ ]:
def detect_R_peaks(dataset):
    """
    Extract the ECG channel from a dataset and detect
    R-peaks using the Pan-Tompkins algorithm.

    Parameters
    ----------
    dataset : BaseDataset object

    Returns
    -------
    rpeaks : ndarray of int - sample indices of detected R-peaks
    """
    ecg = ecg = dataset.raw.get_data(picks='EKG').squeeze()

    #Detect R peaks
    _, info = nk.ecg_peaks (ecg, sampling_rate = 256, method='pantompkins1985')
    rpeaks = info['ECG_R_Peaks']

    return rpeaks
    

In [ ]:
def get_HRV(rpeaks):
    """
    Convert R-peak sample indices to a uniformly-sampled HRV signal.

    Steps:
      1. Convert sample indices to times (seconds) at 256 Hz.
      2. Compute successive RR intervals (beat-to-beat differences).
      3. Cubic-spline interpolate onto a uniform 25 Hz grid.

    Parameters
    ----------
    rpeaks : ndarray of int - R-peak sample indices (at 256 Hz)

    Returns
    -------
    hrv_continuous : ndarray - uniformly sampled HRV signal at 25 Hz
    """
    times = rpeaks / 256

    rr_intervals = np.diff(times)

    rr_times = times[1:]

    fs = 25 #Hz
    t_uniform = np.arange(rr_times[0], rr_times[-1], 1/fs)

    interp_func = interp1d(rr_times, rr_intervals, kind='cubic')
    hrv_continuous = interp_func(t_uniform)

    return hrv_continuous

In [ ]:
def compute_hrv_bands(hrv, fs):
    """
    Decompose HRV into Low-Frequency (LF) and High-Frequency (HF) bands
    using bandpass filters and Hilbert envelope extraction.

    Standard HRV frequency bands:
      LF : 0.04 - 0.15 Hz 
      HF : 0.15 - 0.40 Hz 

    Parameters
    ----------
    hrv : ndarray - HRV signal (already resampled to fs)
    fs  : float   - sampling frequency of hrv in Hz

    Returns
    -------
    lf_envelope : ndarray - amplitude envelope of the LF component
    hf_envelope : ndarray - amplitude envelope of the HF component
    """
    nyq = fs / 2
    
    # LF: 0.04 - 0.15 Hz
    b_lf, a_lf = butter(4, [0.04/nyq, 0.15/nyq], btype='band')
    hrv_lf = filtfilt(b_lf, a_lf, hrv)
    lf_envelope = np.abs(hilbert(hrv_lf))
    
    # HF: 0.15 - 0.4 Hz
    b_hf, a_hf = butter(4, [0.15/nyq, 0.4/nyq], btype='band')
    hrv_hf = filtfilt(b_hf, a_hf, hrv)
    hf_envelope = np.abs(hilbert(hrv_hf))
    
    return lf_envelope, hf_envelope

## VAR Model and GC functions
Functions to fit a Vector AutoRegressive (VAR) model, select the optimal
lag order, and compute Granger Causality (GC) between signals.

In [ ]:
def fit_var(data, p, tau):
    """
    Fit a Vector AutoRegressive (VAR) model of order p with lag spacing tau.

    A VAR model predicts each variable at time t using its own past values
    and those of every other variable. Here we allow non-consecutive lags
    (controlled by tau) to capture slower dynamics.

    Parameters
    ----------
    data : ndarray (n_vars, n_samples)
    p    : int - model order
    tau  : int - lag step

    Returns
    -------
    coeffs    : ndarray (n_vars, n_vars*p) - fitted coefficient matrix
    residuals : ndarray (n_vars, n_effective_samples) - model residuals
    noise_cov : ndarray (n_vars, n_vars) - residual covariance matrix
    """
    n_vars, n_samples = data.shape
    max_lag = p * tau
    Y = data[:, max_lag:]
    X_rows = []
    for lag in range(1, p + 1):
        X_rows.append(data[:, max_lag - lag*tau : n_samples - lag*tau])
    X = np.vstack(X_rows)
    coeffs, _, _, _ = np.linalg.lstsq(X.T, Y.T, rcond=None)
    coeffs = coeffs.T
    residuals = Y - coeffs @ X
    noise_cov = (residuals @ residuals.T) / residuals.shape[1]
    return coeffs, residuals, noise_cov

In [44]:
def select_var_order(data, max_p=20, tau=1):
    """
    Select optimal VAR order using AIC or BIC.
    
    data : ndarray, shape (n_vars, n_samples)
    Returns optimal p and the criterion values for all tested orders.
    """
    n_vars, n_samples = data.shape
    criterion_values = []

    for p in range(1, max_p + 1):
        _, residuals, noise_cov = fit_var(data, p, tau)
        T = residuals.shape[1]
        
        # Log likelihood of multivariate gaussian
        sign, log_det = np.linalg.slogdet(noise_cov)
        log_likelihood = -T/2 * log_det
        
        # Number of free parameters
        n_params = n_vars * n_vars * p
        
        val = -2 * log_likelihood + n_params * np.log(T)
        
        criterion_values.append(val)

    optimal_p = np.argmin(criterion_values) + 1  # +1 because range starts at 1
    return optimal_p, criterion_values


In [ ]:
def compute_gc(data, target_idx, source_indices, p=8, tau=5):
    """Compute GC from sources -> target.

    Parameters
    ----------
    data           : ndarray (n_vars, n_samples)
    target_idx     : int - row index of the target variable
    source_indices : list of int - row indices of the source variables
    p              : int - model order
    tau            : int - lag step

    Returns
    -------
    GC value : float 
    """
    all_idx = sorted(set([target_idx] + list(source_indices)))
    _, _, cov_full = fit_var(data[all_idx], p, tau)
    pos = all_idx.index(target_idx)
    sigma_full = cov_full[pos, pos]

    _, _, cov_reduced = fit_var(data[[target_idx]], p, tau)
    sigma_reduced = cov_reduced[0, 0]

    return max(sigma_reduced - sigma_full, 0.0)

In [ ]:
def compute_subject_gc(eeg_data, ch_names, hrv, fs_eeg, fs_target, tau=5, max_p=20):
    """
    Run the full GC analysis for one subject across all EEG channels.

    For every channel in eeg_ch:
      - Extract delta (0.5-4 Hz) and alpha (8-13 Hz) band envelopes
      - Resample envelopes and HRV bands to fs_target
      - Z-score normalise all signals
      - Select optimal VAR order via BIC
      - Compute bidirectional GC: EEG->HRV/LF/HF and HRV/LF/HF->EEG

    Parameters
    ----------
    eeg_data  : ndarray (n_channels, n_samples)
    ch_names  : list of str - channel names
    hrv       : array-like - HRV signal
    fs_eeg    : float - EEG sampling rate
    fs_target : float - common target sampling rate for GC
    tau       : int - VAR lag step
    max_p     : int - maximum VAR order to test

    Returns
    -------
    electrode_results : dict {channel: {measure: gc_value}}
    """
    hrv = np.array(hrv).flatten()
    

    if fs_target != 25:
        hrv = resample_to(hrv, 25, fs_target)

    lf_power, hf_power = compute_hrv_bands(hrv, fs_target)
    electrode_results = {}

    for ch in eeg_ch:
        if ch not in ch_names:
            continue

        idx =ch_names.index(ch)

        # Get envelope for this single electrode
        delta_env = mean_band_envelope(eeg_data, [idx], fs_eeg, 0.5, 4.0)
        alpha_env = mean_band_envelope(eeg_data, [idx], fs_eeg, 8.0, 13.0)


        delta_rs = resample_to(delta_env, fs_eeg, fs_target)
        alpha_rs = resample_to(alpha_env, fs_eeg, fs_target)

        # Trim
        min_len = min(len(hrv), len(delta_rs), len(alpha_rs), 
                      len(lf_power), len(hf_power))
        hrv   = hrv[:min_len]
        delta_t = delta_rs[:min_len]
        alpha_t = alpha_rs[:min_len]
        lf_t    = lf_power[:min_len]
        hf_t    = hf_power[:min_len]


        # Normalize all signals
        def norm(x):
            return (x - x.mean()) / (x.std() + 1e-12)
        
        hrv_n   = norm(hrv)
        delta_n = norm(delta_t)
        alpha_n = norm(alpha_t)
        lf_n    = norm(lf_t)
        hf_n    = norm(hf_t)

        # Select optimal p once using HRV + delta
        p_opt, _ = select_var_order(
            np.array([hrv_n, delta_n]), max_p, tau)
        
        ch_results = {'optimal_p': p_opt}

        for band_name, eeg_n in [('delta', delta_n), ('alpha', alpha_n)]:
                for target_name, target_n in [('hrv', hrv_n), 
                                            ('lf',  lf_n), 
                                            ('hf',  hf_n)]:
                    seg = np.array([target_n, eeg_n])
                    ch_results[f'{band_name}_eeg_to_{target_name}'] = compute_gc(
                        seg, target_idx=0, source_indices=[1], p=p_opt, tau=tau)
                    ch_results[f'{target_name}_to_{band_name}_eeg'] = compute_gc(
                        seg, target_idx=1, source_indices=[0], p=p_opt, tau=tau)

        electrode_results[ch] = ch_results

    return electrode_results

## Run GC Pipeline Over All Segments
Loop through the dataset, compute GC per subject, and save results to JSON.


In [51]:
epilepsy_summaries  = []
control_summaries = []

skiped_seg = []

fs_eeg = 256  
fs_target = 8

MIN_SAMPLES = 200

for i, d in enumerate (tuh_bhi.datasets):
    print(i)
    eeg_data = d.raw.get_data()
    ch_names = d.raw.ch_names
    rpeaks = detect_R_peaks(d)
    hrv = get_HRV(rpeaks)

    if len(hrv) < MIN_SAMPLES:
            print(f"  Skipping subject {i}: HRV too short ({len(hrv)} samples)")
            skiped_seg.append(i)
            continue

    sub_results = compute_subject_gc (eeg_data, ch_names, hrv, fs_eeg, fs_target)

    if d.description["target"] == 1:
        epilepsy_summaries.append(sub_results)
    else:
        control_summaries.append(sub_results)

0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
51
52
53
54
55
56
57
58
59
60
61
62
63
64
65
66
67
68
69
70
71
72
73
74
75
76
77
78
79
80
81
82
83
84
85
86
87
88
89
90
91
92
93
94
95
96
97
98
99
100
101
102
103
104
105
106
107
108
109
110
111
112
113
114
115
116
117
118
119
120
121
122
123
124
125
126
127
128
129
130
131
132
133
134
135
136
137
138
139
140
141
142
143
144
145
146
147
148
149
150
151
152
153
154
155
156
157
158
159
160
161
162
163
164
165
166
167
168
169
170
171
172
173
174
175
176
177
178
179
180
181
182
183
184
185
186
187
188
189
190
191
192
193
194
195
196
197
198
199
200
201
202
203
204
205
206
207
208
209
210
211
212
213
214
215
216
217
218
219
220
221
222
223
224
225
226
227
228
229
230
231
232
233
234
235
236
237
238
239
240
241
242
243
244
245
246
247
248
249
250
251
252
253
254
255
256
257
258
259
260
261
262
263
264
265
266
267
268
269
270
271
272
273
274
275
276
27

In [52]:

def convert_numpy(obj):
    if isinstance(obj, np.integer):
        return int(obj)
    elif isinstance(obj, np.floating):
        return float(obj)
    elif isinstance(obj, np.ndarray):
        return obj.tolist()
    raise TypeError(f"Object of type {type(obj)} is not JSON serializable")

with open("epilepsy_summaries_1.json", "w") as file:
    json.dump(epilepsy_summaries, file, indent=4, default=convert_numpy)

with open("control_summaries_1.json", "w") as file:
    json.dump(control_summaries, file, indent=4, default=convert_numpy)

with open("skiped_seg_1.json", "w") as file:
    json.dump(skiped_seg, file, indent=4, default=convert_numpy)

In [ ]:
with open("epilepsy_summaries.json", "r") as f:
    epilepsy_summaries = json.load(f)
with open("control_summaries.json", "r") as f:
    control_summaries = json.load(f)

In [ ]:
print(len(epilepsy_summaries))
print(len(control_summaries))

1361
257


In [ ]:
# Inspect the structure of the results for one segment
print(type(epilepsy_summaries[0]))
print(list(epilepsy_summaries[0].keys()))  # electrode names
print(list(epilepsy_summaries[0]['T3'].keys()))  # measure names for one electrode

<class 'dict'>
['FP1', 'FP2', 'F3', 'F4', 'C3', 'C4', 'P3', 'P4', 'O1', 'O2', 'F7', 'F8', 'T3', 'T4', 'T5', 'T6', 'FZ', 'CZ', 'PZ']
['optimal_p', 'delta_eeg_to_hrv', 'hrv_to_delta_eeg', 'delta_eeg_to_lf', 'lf_to_delta_eeg', 'delta_eeg_to_hf', 'hf_to_delta_eeg', 'alpha_eeg_to_hrv', 'hrv_to_alpha_eeg', 'alpha_eeg_to_lf', 'lf_to_alpha_eeg', 'alpha_eeg_to_hf', 'hf_to_alpha_eeg']


## PID Functions

In [ ]:
def compute_pid(data, target_idx, source1_idx, source2_idx, p, tau):
    """
    Compute PID measures for two sources predicting a target.
    
    Returns unique, redundant and synergistic GC.
    
    Based on Faes et al. 2017 linear PID formulation:
    the same approach used in Pernice et al. 2022.
    """
    # Bivariate GC from each source individually
    # Source 1 alone
    idx_s1 = sorted([target_idx, source1_idx])
    _, _, cov_s1_full    = fit_var(data[idx_s1], p, tau)
    _, _, cov_target     = fit_var(data[[target_idx]], p, tau)
    pos_t_in_s1 = idx_s1.index(target_idx)
    
    F_s1 = max(float(cov_target[0,0] - cov_s1_full[pos_t_in_s1, pos_t_in_s1]), 0.0)
    
    # Source 2 alone
    idx_s2 = sorted([target_idx, source2_idx])
    _, _, cov_s2_full = fit_var(data[idx_s2], p, tau)
    pos_t_in_s2 = idx_s2.index(target_idx)
    
    F_s2 = max(float(cov_target[0,0] - cov_s2_full[pos_t_in_s2, pos_t_in_s2]), 0.0)
    
    # Joint GC from both sources together
    idx_joint = sorted([target_idx, source1_idx, source2_idx])
    _, _, cov_joint_full = fit_var(data[idx_joint], p, tau)
    pos_t_in_joint = idx_joint.index(target_idx)
    
    F_joint = max(float(cov_target[0,0] - cov_joint_full[pos_t_in_joint, pos_t_in_joint]), 0.0)
    
    # PID decomposition (Barrett 2015 / Faes 2017 linear formulation)
    # Redundancy = minimum individual GC
    R = min(F_s1, F_s2)
    
    # Unique GCs
    U_s1 = F_s1 - R
    U_s2 = F_s2 - R
    
    # Synergy = what joint adds beyond the maximum individual
    S = max(F_joint - max(F_s1, F_s2), 0.0)
    
    return {
        'F_s1':       F_s1,      # bivariate GC source 1
        'F_s2':       F_s2,      # bivariate GC source 2
        'F_joint':    F_joint,   # joint GC
        'unique_s1':  U_s1,      # unique to source 1
        'unique_s2':  U_s2,      # unique to source 2
        'redundant':  R,         # shared between sources
        'synergistic': S         # joint effect
    }


def compute_subject_pid(eeg_data, ch_names, hrv, fs_eeg,
                         fs_target=8, tau=5, max_p=20):
    """
    Compute PID for hemisphere pairs.
    Returns dict: {pair_name: {measure: value}}
    """
    hrv = np.array(hrv).flatten()
    hrv = resample_to(hrv, 25, fs_target)


    electrode_pairs = [
        ('T3', 'T4'),
        ('F7', 'F8'),
        ('O1', 'O2'),
        ('T5', 'T6'),
    ]

    pair_results = {}

    for ch_left, ch_right in electrode_pairs:
        if ch_left not in ch_names or ch_right not in ch_names:
            continue

        idx_l = ch_names.index(ch_left)
        idx_r = ch_names.index(ch_right)

        # Get envelopes
        for band_name, lowcut, highcut in [('delta', 0.5, 4.0),
                                            ('alpha', 8.0, 13.0)]:
            env_l = mean_band_envelope(eeg_data, [idx_l], fs_eeg, lowcut, highcut)
            env_r = mean_band_envelope(eeg_data, [idx_r], fs_eeg, lowcut, highcut)

            env_l_8 = resample_to(env_l, fs_eeg, fs_target)
            env_r_8 = resample_to(env_r, fs_eeg, fs_target)

            min_len = min(len(hrv), len(env_l_8), len(env_r_8))

            if min_len < 200:
                continue

            def norm(x):
                return (x - x.mean()) / (x.std() + 1e-12)

            hrv_n  = norm(hrv[:min_len])
            left_n = norm(env_l_8[:min_len])
            right_n = norm(env_r_8[:min_len])

            # Stack as [HRV, left, right]
            # indices: HRV=0, left=1, right=2
            seg = np.array([hrv_n, left_n, right_n])

            p_opt, _ = select_var_order(
                np.array([hrv_n, left_n]), max_p, tau
            )

            pid = compute_pid(
                seg,
                target_idx=0,    # HRV is target
                source1_idx=1,   # left hemisphere
                source2_idx=2,   # right hemisphere
                p=p_opt,
                tau=tau
            )

            pair_key = f"{ch_left}_{ch_right}_{band_name}"
            pair_results[pair_key] = pid

    return pair_results

In [ ]:
def compute_subject_pid_cardiac_to_brain(eeg_data, ch_names, hrv, fs_eeg,
                                          fs_target=8, tau=5, max_p=20):
    """
    PID decomposition with alpha EEG as target, LF and HF as sources.
    
    For each electrode:
    sources: LF, HF -> target: alpha_EEG
    Decomposes how LF and HF jointly/uniquely/redundantly predict alpha EEG.
    """
    hrv = np.array(hrv).flatten()
    hrv = resample_to(hrv, 25, fs_target)
    lf_power, hf_power = compute_hrv_bands(hrv, fs_target)

    electrode_results = {}

    for ch in eeg_ch:
        if ch not in ch_names:
            continue

        idx = ch_names.index(ch)

        # Get alpha envelope for this electrode
        alpha_env = mean_band_envelope(eeg_data, [idx], fs_eeg, 8.0, 13.0)
        alpha_rs  = resample_to(alpha_env, fs_eeg, fs_target)

        # Trim
        min_len = min(len(hrv), len(alpha_rs),
                      len(lf_power), len(hf_power))

        if min_len < 200:
            continue

        alpha_t = alpha_rs[:min_len]
        lf_t    = lf_power[:min_len]
        hf_t    = hf_power[:min_len]

        def norm(x):
            return (x - x.mean()) / (x.std() + 1e-12)

        alpha_n = norm(alpha_t)
        lf_n    = norm(lf_t)
        hf_n    = norm(hf_t)

        # Select optimal p using alpha + LF
        seg_for_order = np.array([alpha_n, lf_n])
        p_opt, _ = select_var_order(seg_for_order, max_p, tau)

        # Stack as [alpha_EEG, LF, HF]
        # target = alpha_EEG (idx 0)
        # source1 = LF (idx 1)
        # source2 = HF (idx 2)
        seg = np.array([alpha_n, lf_n, hf_n])

        pid = compute_pid(
            seg,
            target_idx=0,   # alpha EEG is target
            source1_idx=1,  # LF is source 1
            source2_idx=2,  # HF is source 2
            p=p_opt,
            tau=tau
        )

        # Also add delta band for comparison
        delta_env = mean_band_envelope(eeg_data, [idx], fs_eeg, 0.5, 4.0)
        delta_rs  = resample_to(delta_env, fs_eeg, fs_target)
        delta_t   = delta_rs[:min_len]
        delta_n   = norm(delta_t)

        seg_delta = np.array([delta_n, lf_n, hf_n])
        pid_delta = compute_pid(
            seg_delta,
            target_idx=0,
            source1_idx=1,
            source2_idx=2,
            p=p_opt,
            tau=tau
        )

        electrode_results[ch] = {
            'alpha': pid,
            'delta': pid_delta,
            'optimal_p': p_opt
        }

    return electrode_results


def flatten_pid_cardiac(electrode_results):
    """
    Average per-electrode PID values across all electrodes into a single
    flat dict.

    Parameters
    ----------
    electrode_results : dict {electrode: {'alpha': pid_dict, 'delta': pid_dict}}

    Returns
    -------
    flat : dict {measure_name: mean_value}
    """
    flat = {}
    electrodes = list(electrode_results.keys())
    
    if not electrodes:
        return flat

    for band in ['alpha', 'delta']:
        for pid_key in ['unique_s1', 'unique_s2', 'redundant', 'synergistic',
                        'F_s1', 'F_s2', 'F_joint']:
            
            # Rename s1/s2 to lf/hf
            label = pid_key.replace('s1', 'lf').replace('s2', 'hf')
            measure_name = f'{band}_{label}'
            
            vals = []
            for ch in electrodes:
                if band in electrode_results[ch] and \
                   pid_key in electrode_results[ch][band]:
                    vals.append(electrode_results[ch][band][pid_key])
            
            if vals:
                flat[measure_name] = np.mean(vals)
    
    return flat

## Run PID Pipeline Over All Subjects
First PID with electrode pairs as sources.
Second PID with LF and HF as sources.

In [54]:
pid_epilepsy_summaries  = []
pid_control_summaries = []

pid_skiped_seg = []

fs_eeg = 256  
fs_target = 8

MIN_SAMPLES = 200

for i, d in enumerate (tuh_bhi.datasets):
    print(i)
    eeg_data = d.raw.get_data()
    ch_names = d.raw.ch_names
    rpeaks = detect_R_peaks(d)
    hrv = get_HRV(rpeaks)

    if len(hrv) < MIN_SAMPLES:
            print(f"  Skipping subject {i}: HRV too short ({len(hrv)} samples)")
            pid_skiped_seg.append(i)
            continue

    sub_results = compute_subject_pid (eeg_data, ch_names, hrv, fs_eeg, fs_target)

    if d.description["target"] == 1:
        pid_epilepsy_summaries.append(sub_results)
    else:
        pid_control_summaries.append(sub_results)

0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
51
52
53
54
55
56
57
58
59
60
61
62
63
64
65
66
67
68
69
70
71
72
73
74
75
76
77
78
79
80
81
82
83
84
85
86
87
88
89
90
91
92
93
94
95
96
97
98
99
100
101
102
103
104
105
106
107
108
109
110
111
112
113
114
115
116
117
118
119
120
121
122
123
124
125
126
127
128
129
130
131
132
133
134
135
136
137
138
139
140
141
142
143
144
145
146
147
148
149
150
151
152
153
154
155
156
157
158
159
160
161
162
163
164
165
166
167
168
169
170
171
172
173
174
175
176
177
178
179
180
181
182
183
184
185
186
187
188
189
190
191
192
193
194
195
196
197
198
199
200
201
202
203
204
205
206
207
208
209
210
211
212
213
214
215
216
217
218
219
220
221
222
223
224
225
226
227
228
229
230
231
232
233
234
235
236
237
238
239
240
241
242
243
244
245
246
247
248
249
250
251
252
253
254
255
256
257
258
259
260
261
262
263
264
265
266
267
268
269
270
271
272
273
274
275
276
27

In [61]:
pid_epilepsy_summaries = []
pid_control_summaries  = []

for i, d in enumerate(tuh_bhi.datasets):
    print(i)
    try:
        eeg_data = d.raw.get_data()
        ch_names = d.raw.ch_names
        rpeaks   = detect_R_peaks(d)
        hrv      = np.array(get_HRV(rpeaks)).flatten()

        pid_results = compute_subject_pid_cardiac_to_brain(
            eeg_data, ch_names, hrv,
            fs_eeg=256, fs_target=8, tau=5
        )

        if not pid_results:
            continue

        flat = flatten_pid_cardiac(pid_results)

        if d.description["target"] == 1:
            pid_epilepsy_summaries.append(flat)
        else:
            pid_control_summaries.append(flat)

    except Exception as e:
        print(f"  Subject {i} failed: {e}")
        continue

print(f"\nEpilepsy: {len(pid_epilepsy_summaries)}")
print(f"Control:  {len(pid_control_summaries)}")

0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
51
52
53
54
55
56
57
58
59
60
61
62
63
64
65
66
67
68
69
70
71
72
73
74
75
76
77
78
79
80
81
82
83
84
85
86
87
88
89
90
91
92
93
94
95
96
97
98
99
100
101
102
103
104
105
106
107
108
109
110
111
112
113
114
115
116
117
118
119
120
121
122
123
124
125
126
127
128
129
130
131
132
133
134
135
136
137
138
139
140
141
142
143
144
145
146
147
148
149
150
151
152
153
154
155
156
157
158
159
160
161
162
163
164
165
166
167
168
169
170
171
172
173
174
175
176
177
178
179
180
181
182
183
184
185
186
187
188
189
190
191
192
193
194
195
196
197
198
199
200
201
202
203
204
205
206
207
208
209
210
211
212
213
214
215
216
217
218
219
220
221
222
223
224
225
226
227
228
229
230
231
232
233
234
235
236
237
238
239
240
241
242
243
244
245
246
247
248
249
250
251
252
253
254
255
256
257
258
259
260
261
262
263
264
265
266
267
268
269
270
271
272
273
274
275
276
27

In [62]:
def convert_numpy(obj):
    if isinstance(obj, np.integer):
        return int(obj)
    elif isinstance(obj, np.floating):
        return float(obj)
    elif isinstance(obj, np.ndarray):
        return obj.tolist()
    raise TypeError(f"Object of type {type(obj)} is not JSON serializable")

with open("pid_epilepsy_summaries_2.json", "w") as file:
    json.dump(pid_epilepsy_summaries, file, indent=4, default=convert_numpy)

with open("pid_control_summaries_2.json", "w") as file:
    json.dump(pid_control_summaries, file, indent=4, default=convert_numpy)


## Test GC pipeline for one segment

In [ ]:
skiped_seg = []

fs_eeg = 256  
fs_target = 8

test = tuh_bhi.datasets[1]
eeg_data = d.raw.get_data()
ch_names = d.raw.ch_names
rpeaks = detect_R_peaks(d)
hrv = get_HRV(rpeaks)

seg_results = compute_subject_gc (eeg_data, ch_names, hrv, fs_eeg, fs_target)

In [ ]:
print (seg_results)

{'FP1': {'optimal_p': np.int64(20), 'delta_eeg_to_hrv': np.float64(0.0010631002248984306), 'hrv_to_delta_eeg': np.float64(0.0025870688593550017), 'delta_eeg_to_lf': np.float64(3.108602565132161e-06), 'lf_to_delta_eeg': np.float64(0.00783147837837539), 'delta_eeg_to_hf': np.float64(0.0002745495935363544), 'hf_to_delta_eeg': np.float64(0.004717031431648144), 'alpha_eeg_to_hrv': np.float64(0.0003505128812652872), 'hrv_to_alpha_eeg': np.float64(0.003910284655527674), 'alpha_eeg_to_lf': np.float64(1.8325760038928629e-06), 'lf_to_alpha_eeg': np.float64(0.0064519701081026115), 'alpha_eeg_to_hf': np.float64(0.00022623218966442982), 'hf_to_alpha_eeg': np.float64(0.002795682068852612)}, 'FP2': {'optimal_p': np.int64(20), 'delta_eeg_to_hrv': np.float64(0.0005289790202782155), 'hrv_to_delta_eeg': np.float64(0.007265625520527896), 'delta_eeg_to_lf': np.float64(1.8555227168272218e-06), 'lf_to_delta_eeg': np.float64(0.005156498292552603), 'delta_eeg_to_hf': np.float64(0.0002170122665060316), 'hf_to_d